# Named Segments and Business-Aligned Rewards-Program Proposal

This notebook adds presentation names and proposed member benefits to the consolidated customer export. It does not rerun clustering. It keeps evidence-based descriptions alongside the names, and records commercial assumptions explicitly.

The earlier Elena-perk assignment policy is preserved in earlier files but is not treated as the final business recommendation. New benefits remain draft proposals requiring service capability, partner, budget and controlled-response validation. The output does not establish favorite perks or claim verified low cost.

Inputs: the study feature table and notebook 08's consolidated customer export.

Outputs: `reports/business_aligned_delivery/`. Source files and previous exports are unchanged. Repeated runs replace only this notebook's derived outputs. Dependencies: pandas.


In [ ]:
from pathlib import Path
import pandas as pd

print("STEP 1: Load the consolidated segments and study features", flush=True)
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "reports" / "consolidated_segmentation" / "customer_segments_and_perks_draft.csv").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Run notebook 08 first. No output written.")
BASE = PROJECT / "reports" / "consolidated_segmentation" / "customer_segments_and_perks_draft.csv"
SOURCE = PROJECT / "data" / "own_processed" / "user_features_windowed.csv"
OUT = PROJECT / "reports" / "business_aligned_delivery"
a = pd.read_csv(BASE,dtype={"user_id":"string"})
f = pd.read_csv(SOURCE,usecols=["user_id","num_hotel_trips","num_flight_trips"],dtype={"user_id":"string"})
assert a.user_id.is_unique and f.user_id.is_unique and set(a.user_id)==set(f.user_id)
# Preserve the prior policy as an archive; do not mix its rules with the agreed proposal.
previous_policy_columns = ["proposed_perk","perk_status","assignment_reason","evidence_confidence",
    "assignment_is_fallback","eligible_rule_count","eligible_perks","priority_resolved_overlap","preference_verified"]
a = a.drop(columns=previous_policy_columns)
a = a.merge(f,on="user_id",validate="one_to_one")
names = {"S1":"The Regulars","S2":"The Reconnectors","S3":"The Explorers","S4":"The Occasional Visitors"}
a = a.rename(columns={"segment_name":"segment_description"})
a["segment_name"] = a.segment_id.map(names)
assert a.segment_name.notna().all() and set(a.segment_id)==set(names)
a["segment_name_status"] = "Illustrative relative-activity label; identity and motivation unverified"

print("STEP 2: Apply the agreed business proposal, not the old Elena-perk rules", flush=True)
policy = {
 "S1": {"benefit":"Availability-based partner stay enhancement",
        "objective":"Maintain the observed booking relationship",
        "hypothesis":"A convenient stay enhancement may make membership relevant to repeat bookers using hotels.",
        "cost_control":"Participating properties, approved dates and benefit types, availability limits, negotiated funding and redemption caps.",
        "feasibility":"Partner agreement and fulfillment process must be verified; spare capacity is not costless.",
        "confidence":"Observed booking and hotel applicability; preference and effectiveness unverified"},
 "S2": {"benefit":"Capped next-booking partner credit",
        "objective":"Test a relevant reason for previous bookers to return",
        "hypothesis":"A bounded future-booking benefit may encourage enrollment and repeat booking; subsidy of existing demand remains a risk.",
        "cost_control":"Negotiated partner cost-sharing, fixed pilot budget, qualifying bookings and capped reimbursement.",
        "feasibility":"Funding, qualifying terms, reimbursement and cancellation handling must be approved.",
        "confidence":"Activity pattern supports a communication test; benefit preference unverified"},
 "S3": {"benefit":"Member trip tracker: saved options and useful alerts",
        "objective":"Make membership useful during current travel planning",
        "hypothesis":"Planning tools may reduce effort and encourage return visits before a monetary subsidy is offered.",
        "cost_control":"Assess build and operating costs; potentially low marginal delivery cost after development, not zero cost.",
        "feasibility":"Confirm capabilities, notification consent and member value beyond existing search features.",
        "confidence":"Recent engagement and lower booking activity observed; planning friction unverified"},
 "S4": {"benefit":"Preference-led member welcome: choose an approved benefit",
        "objective":"Learn what would make membership relevant before funding larger incentives",
        "hypothesis":"A lightweight invitation can elicit preference and test engagement without claiming churn or known needs.",
        "cost_control":"Offer only approved, budgeted choices with transparent eligibility; avoid automatic high-cost rewards.",
        "feasibility":"A usable preference interface and real approved choices must exist before launch.",
        "confidence":"Preference is explicitly unknown; invitation hypothesis only"}}
for output,key in [("proposed_member_benefit","benefit"),("communication_objective","objective"),
                   ("customer_value_hypothesis","hypothesis"),("cost_control_conditions","cost_control"),
                   ("feasibility_conditions","feasibility"),("evidence_basis_and_limits","confidence")]:
    a[output] = a.segment_id.map(lambda s:policy[s][key])
# Hotel-specific proposals are not silently sent to customers with no observed hotel use.
a["benefit_applicability_fallback"] = a.segment_id.eq("S1") & a.num_hotel_trips.eq(0)
fallback = a.benefit_applicability_fallback
for output,key in [("proposed_member_benefit","benefit"),("customer_value_hypothesis","hypothesis"),
                   ("cost_control_conditions","cost_control"),("feasibility_conditions","feasibility")]:
    a.loc[fallback,output] = policy["S3"][key]
a.loc[fallback,"evidence_basis_and_limits"] = "No observed hotel booking; planning-service alternative is low-confidence and unverified"
a["benefit_preference_verified"] = False
a["partner_agreement_verified"] = False
a["benefit_cost_estimated"] = False
a["recommendation_status"] = "Agreed draft business proposal; feasibility and effect require validation"
assert a.proposed_member_benefit.notna().all() and len(a)==len(f)
assert not (a.num_hotel_trips.eq(0) & a.proposed_member_benefit.eq(policy["S1"]["benefit"])).any()

print("STEP 3: Save the business-aligned export and presentation materials", flush=True)
OUT.mkdir(parents=True,exist_ok=True)
a.sort_values("user_id").to_csv(OUT / "customer_segments_business_aligned.csv",index=False)
summary = a.groupby(["segment_id","segment_name","segment_description","proposed_member_benefit"],dropna=False).size().reset_index(name="customers")
summary["share_of_segment"] = summary.customers / summary.segment_id.map(a.segment_id.value_counts())
assert summary.customers.sum()==len(a)
summary.to_csv(OUT / "segment_benefit_summary.csv",index=False)
profiles = a.groupby(["segment_id","segment_name","segment_description"]).agg(
    customers=("user_id","size"), median_engagement_recency_days=("session_recency_days","median"),
    median_recent_noncanceled_trips=("rfm_frequency_trips","median"),
    known_recent_booking_value_customers=("rfm_monetary_estimate","count"),
    median_known_recent_booking_value=("rfm_monetary_estimate","median")).reset_index()
profiles.to_csv(OUT / "named_segment_profiles.csv",index=False)
lines = ["# Named Customer Segments and Business-Aligned Program Proposal", "",
         "## Business story", "TravelTide plans to introduce a rewards program. The proposed strategy is to turn its search and inventory advantage into repeat customer relationships through useful member services and carefully negotiated benefits rather than blanket subsidies. The brief's retention problem supplies the business context; cluster patterns do not prove causes of retention.",
         "", f"## Coverage", f"All {len(a):,} customers retain their existing four-group K-means assignments. No clustering was rerun. The prior Elena-perk policy is superseded for this business proposal but preserved in earlier outputs.",
         "", "## Named segments", "```", profiles[["segment_id","segment_name","customers","median_engagement_recency_days","median_recent_noncanceled_trips"]].to_string(index=False), "```",
         "Segment names are illustrative labels for relative activity patterns—not verified traveler identities, motivations or preferences. The Reconnectors describes a communication opportunity, not an intention to return. The Occasional Visitors is not an estimated visit-frequency category; it represents relatively lower booking activity and less recent engagement.",
         "", "## Proposal and economic conditions"]
for seg in ["S1","S2","S3","S4"]:
    row = policy[seg]
    lines += [f"### {seg}: {names[seg]}", f"- Business objective: {row['objective']}.",
              f"- Proposed benefit: {row['benefit']}.", f"- Customer-value hypothesis: {row['hypothesis']}",
              f"- Cost controls: {row['cost_control']}", f"- Feasibility conditions: {row['feasibility']}"]
lines += [f"", f"Hotel-applicability fallback: {int(fallback.sum()):,} S1 customers without observed hotel bookings receive the planning-service proposal instead. This is an applicability safeguard, not a newly discovered segment.",
          "", "## Evidence and pilot safeguards", "Observed activity suggests communication objectives, not proven needs. Benefits have no verified preferences, agreements, costs or causal response estimates. Access to inventory does not establish ownership or free fulfillment. Partner cost-sharing must be commercially feasible and does not eliminate total system cost. No profit or retention uplift is promised. Monetary value remains estimated booking expenditure with explicit missingness; it is not margin.",
          "", "## What changed", "Presentation names are now separate from evidence-based descriptions. The model and its assignments are unchanged. Proposed benefits now follow the agreed business strategy and cost conditions rather than the earlier threshold hierarchy. This does not make the new proposals empirically optimal; customer and commercial validation remain necessary.",
          "", "## Test design", "First verify approved service capabilities, partner funding, eligibility, liability caps and fulfillment. Then randomize customer-level invitations between the proposed relevant message and a generic invitation or approved alternative. Track enrollment, subsequent repeat bookings and cost. Analyze all randomized customers, not only email openers. Evaluate incremental contribution only when margin and cost inputs are available.",
          "", "## Deliverables", "customer_segments_business_aligned.csv contains one row per customer with both names and descriptions, proposed member value and economic/feasibility conditions. segment_benefit_summary.csv provides reconciled counts. named_segment_profiles.csv preserves original-unit profiles. presentation_story_cards.md gives three illustrative segment narratives."]
(OUT / "business_delivery_summary.md").write_text("\n".join(lines),encoding="utf-8")
cards = ["# Presentation Story Cards", "", "These are illustrative composites grounded in cluster medians, not real individual customers. The business objective, proposed need and effect are hypotheses. Mention all four segments in the overview; use the following three for a deeper discussion."]
for seg in ["S1","S2","S3"]:
    row = profiles.loc[profiles.segment_id.eq(seg)].iloc[0]
    offer = policy[seg]
    cards += ["",f"## {seg}: {names[seg]}",
              f"**Observed pattern:** {int(row.customers):,} customers; median engagement recency {row.median_engagement_recency_days:g} days and {row.median_recent_noncanceled_trips:g} noncanceled trips in the recent booking window.",
              f"**Business opportunity:** {offer['objective']}.",
              f"**Illustrative customer moment:** An example customer with this relative activity pattern may find the following value relevant. This is not a psychological claim or a verified individual profile.",
              f"**Proposed member value:** {offer['benefit']}.",
              f"**Why investigate it:** {offer['hypothesis']}",
              f"**Economic conditions:** {offer['cost_control']}",
              f"**Do not promise before checking:** {offer['feasibility']}",
              "**Pilot question:** Does this approved message increase program enrollment and subsequent incremental booking relative to a generic invitation, at an acceptable expected cost?"]
cards += ["", "## S4: The Occasional Visitors — overview coverage", "Retain this segment in the group overview and delivery table. Propose a lightweight benefit-choice invitation rather than an expensive automatic reward. Do not infer dormancy, churn or low customer worth from the label.",
          "", "## Concise presentation sequence", "Business context and program launch → cohort and model choice → all four named segments → three illustrative customer moments → partner feasibility and cost controls → pilot and limitations. The analytical model uses engagement recency and booking frequency; RFM/LRFM remain comparison and profiling frameworks."]
(OUT / "presentation_story_cards.md").write_text("\n".join(cards),encoding="utf-8")
print("COMPLETE:",OUT,flush=True)
print("Earlier customer exports, source CSVs and clustering assignments were not modified.",flush=True)
